# 25 - Pass 11 E1: does the MLM gain replicate when the label is not the outlet? (one commit, GPU)

**The question.** On News Headlines the label is the outlet (*The Onion* vs *HuffPost*), so in-domain MLM pretraining may mostly teach outlet style.
iSarcasmEval labels are written by the tweet's author. This notebook repeats the low-label comparison **in-domain on iSarcasmEval** (train on its training tweets, score on its test set)
and adds a **size-matched control** so a small result cannot be blamed on a small pretraining pool.

| Block | Runs | What it answers |
|---|---|---|
| **A1** (BERT), label counts N = 228 / 780 / 3,120, seeds 0-9 [tag `isc`] | 30 | the baseline (228 = the label count of Headlines 1%; 3,120 = every non-dev tweet) |
| **A20** from an MLM encoder pretrained on the iSarcasmEval training tweets (no labels, dev and test excluded), 1,400 steps = 28.7 passes [`isc28`] | 30 | the in-domain gain (H11.8, H11.9) |
| **A20** from the Headlines MLM encoder of Pass 8a [`hs`] | 30 | cross-domain control: does pretraining on headlines help tweets? (H11.10) |
| **A20** from a 4,900-step (100-pass) encoder [`isc100`], N = 228 / 780 | 20 | sensitivity to the pretraining length (H11.12) |
| **A20 on Headlines** (1% and 10%, seeds 0-4) from an encoder pretrained on a random **3,120-headline** subset, 1,400 steps [`hsub`] | 10 | size-matched control (H11.11) |

**120 runs plus three short pretrainings, about 2 GPU-hours (about 1.2 h on T4 x2; estimate).** Primary metric: **ROC-AUC on the iSarcasmEval test set** (14% sarcastic); macro-F1 secondary.
Resumable: if the session stops, run the run cell again.

**Pre-registration.** H11.8-H11.12 in `vault/Pass 11 Predictions.md` (frozen before this code was written), including the reading rule: *replicates* = AUC gain of at least +3, robust, at N = 228 and 780;
*collapses* = gain below +1 or interval including 0 at both. The notebook refuses to start unless that file says FROZEN.

**Before you run, in the right-hand panel:**
- **Input 1:** the same `gainsplit-processed` dataset as notebooks 22-24.
- **Input 2:** the output of **notebook 15** (it holds the seed-0 Headlines MLM encoder, `ckpt/mlm`).
- **Internet: on**, **Accelerator: GPU T4 x2**, then **Save Version -> Save & Run All (Commit)**.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time, json, shutil, threading
from pathlib import Path
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about three minutes)
If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Pre-registration check

In [ ]:
REQUIRE_FROZEN = True
pred = Path(f"{REPO_DIR}/vault/Pass 11 Predictions.md").read_text(encoding="utf-8")
status = [l.split(":", 1)[1].strip() for l in pred.splitlines()[:8] if l.startswith("status:")]
print("predictions status:", status, "| E1 hypotheses present:", "H11.12" in pred)
if REQUIRE_FROZEN:
    assert status == ["FROZEN"] and "H11.12" in pred, "predictions must be FROZEN in this commit"

## 4. Find the Headlines MLM encoder from notebook 15
Used for the cross-domain control (`hs`). The cell compares its weight hashes with the committed receipt and stops if it is missing or different.

In [ ]:
from gainsplit.ssl_pretrain import find_encoder
receipt = json.loads(Path(f"{REPO_DIR}/results/pass8_mlm/pretrain_info.json").read_text())
ENC = str(find_encoder("/kaggle/input", "mlm", seed=0, steps=10000, expected_sha256=receipt["weights_sha256"]))
print("Headlines MLM encoder (seed 0, 10,000 steps):", ENC, "- weights match the Pass 8a receipt")

## 5. Data and the iSarcasmEval label-count budgets
The iSarcasmEval training set is split as in Pass 3 (a fixed 10% stratified dev split, never trained on). This cell draws **nested label sets of N = 228 / 780 / 3,120 tweets per seed (0-9)** from the non-dev tweets
(class share kept, never the dev split; checked) and writes them to `/kaggle/working/extra/`, read through `GAINSPLIT_ISARC_BUDGETS`. The frozen data folder is not touched. The file goes into the results zip as a receipt.

In [ ]:
import torch
from gainsplit.data import (find_data_dir, verify, verify_pass3, make_isarc_budgets, load_isarcasm_splits, load_splits,
                            ISARC_BUDGETS_ENV)
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
verify_pass3(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json and manifest_pass3.json")
from gainsplit.plans import PASS11_E1_BUDGETS, PASS11_E1_SEEDS
budgets = make_isarc_budgets(DATA_DIR, PASS11_E1_BUDGETS, PASS11_E1_SEEDS)
Path("/kaggle/working/extra").mkdir(parents=True, exist_ok=True)
ISARC_PATH = "/kaggle/working/extra/budgets_isarc.json"
Path(ISARC_PATH).write_text(json.dumps(budgets))
os.environ[ISARC_BUDGETS_ENV] = ISARC_PATH
full = load_isarcasm_splits(DATA_DIR)
print(f"iSarcasmEval train: {len(full.labeled)} non-dev tweets ({full.labeled['label'].mean():.3f} sarcastic), dev {len(full.dev)}, test {len(full.isarc_test)} ({full.isarc_test['label'].mean():.3f} sarcastic)")
for n in PASS11_E1_BUDGETS:
    sp = load_isarcasm_splits(DATA_DIR, n, 0)
    print(f"N={n}: seed 0 -> {len(sp.labeled)} tweets, sarcastic share {sp.labeled['label'].mean():.3f}")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 6. The list of runs

In [ ]:
from gainsplit.plans import PLANS, PASS11_E1_CKPT, PASS11_E1_STEPS
PLAN = "pass11_isarc"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
configs = PLANS["pass11_isarc"](ENC, PASS11_E1_CKPT)
print(len(configs), "runs;", sorted({(c.tag, c.train_data) for c in configs}))
for c in configs[:3] + configs[-3:]:
    print(f"{c.run_id:26s} arm={c.arm:3s} data={c.train_data:9s} labels={c.budget} seed={c.seed} init={c.init_dir or '-'}")
# CPU pre-flight: every run resolves to a label set; every run is test-scored (nothing is tuned here)
for c in configs:
    if c.train_data == "isarcasm":
        load_isarcasm_splits(DATA_DIR, c.budget, c.seed)
    else:
        load_splits(DATA_DIR, c.budget, c.seed)
assert all(c.eval_test for c in configs)

## 7. Pretrain three short MLM encoders (about 15 minutes)
No labels are used. Pass 8a's settings (batch 64, lr 1e-4, seed 0) except the texts and the number of steps:
`isc28`: the iSarcasmEval training tweets minus dev, 1,400 steps (28.7 passes); `isc100`: the same, 4,900 steps (100 passes); `hsub`: a random 3,120-headline subset of the Headlines training texts
(fixed subset seed), 1,400 steps. Two GPUs: one runs `isc28` then `hsub`, the other `isc100`. Finished pretrainings are skipped on a restart; receipts go into the zip.

In [ ]:
PRETRAIN = {"isc28": {"seed": 0, "steps": PASS11_E1_STEPS["isc28"], "text_source": "isarc"},
            "isc100": {"seed": 0, "steps": PASS11_E1_STEPS["isc100"], "text_source": "isarc"},
            "hsub": {"seed": 0, "steps": PASS11_E1_STEPS["hsub"], "text_source": "headlines_subset", "subset_n": 3120}}

def pretrain_chain(gpu, tags):
    for tag in tags:
        env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src", CUDA_VISIBLE_DEVICES=str(gpu))
        with open(f"{OUT_DIR}/pretrain_{tag}.log", "w") as log:
            p = subprocess.run([sys.executable, "-m", "gainsplit.ssl_pretrain", "--objective", "mlm", "--data", str(DATA_DIR),
                                "--out", PASS11_E1_CKPT[tag], "--config", json.dumps(PRETRAIN[tag])],
                               stdout=log, stderr=subprocess.STDOUT, env=env)
        assert p.returncode == 0, f"pretraining {tag} failed: see {OUT_DIR}/pretrain_{tag}.log"
        shutil.copy(f"{PASS11_E1_CKPT[tag]}/pretrain_info.json", f"{OUT_DIR}/pretrain_info_{tag}.json")

t0 = time.time()
chains = {0: ["isc28", "hsub"], 1: ["isc100"]} if torch.cuda.device_count() > 1 else {0: ["isc28", "hsub", "isc100"]}
errors = []
def _run(g, tags):
    try:
        pretrain_chain(g, tags)
    except Exception as e:
        errors.append(repr(e))
threads = [threading.Thread(target=_run, args=(g, t)) for g, t in chains.items()]
for t in threads: t.start()
for t in threads: t.join()
assert not errors, errors
for tag in PRETRAIN:
    info = json.loads(Path(f"{OUT_DIR}/pretrain_info_{tag}.json").read_text())
    print(f"{tag}: {info['minutes']} min; {info['n_texts']} texts ({info['text_source']}); dev MLM loss {info['dev_before']['loss']} -> {info['dev_after']['loss']}")
print(f"pretraining: {(time.time() - t0) / 60:.1f} min")

## Run
One run per GPU; when a GPU frees up, the next starts. Logs in `results/pass11_isarc/logs/<run_id>.log`. If the session stops halfway, run this cell again.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## Analysis (CPU, about 2 minutes)
Paired bootstrap over the iSarcasmEval test items (10,000 resamples, same resample for every seed, mean over seeds 0-9), Holm inside each table.
Headlines runs of A1 (Pass 1) and A20 (Pass 8a) for the size-matched control come from the committed results folders.

In [ ]:
from gainsplit import analysis_pass11_isarc as a11i
import pandas as pd
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 140)
RESULTS_DIRS = [d for d in [f"{REPO_DIR}/results/{p}" for p in ("pass1", "pass8_mlm")] if Path(d).exists()] + [OUT_DIR]
print("results folders:", RESULTS_DIRS)
tables = a11i.run_all(RESULTS_DIRS, f"{OUT_DIR}/analysis")
for name in ("summary", "isarc_auc", "isarc_f1", "size_control", "verdicts"):
    print(f"\n=== {name}")
    print(tables[name].to_string())

## Download
Zips the results folder. A commit keeps it in the Output tab: download `results_pass11_isarc.zip` and send it with the notebook log.

In [ ]:
shutil.copy(ISARC_PATH, f"{OUT_DIR}/budgets_isarc.json")
print(shutil.make_archive("/kaggle/working/results_pass11_isarc", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")